# 🚀 OmniVoice Studio — Kaggle Batch Generator (v1 to v5)
Run all cells sequentially in Kaggle with **GPU T4** enabled.

In [ ]:
# ⚙️ Step 1: Check GPU Status
!nvidia-smi
import torch
print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 📦 Step 2: Clone & Install OmniVoice Studio
!pip install -q uv
!git clone https://github.com/debpalash/OmniVoice-Studio.git /kaggle/working/omnivoice-studio
%cd /kaggle/working/omnivoice-studio
!uv sync

In [ ]:
# ⚡ Step 3: Launch OmniVoice Backend Server in Background
import os, time, subprocess
%cd /kaggle/working/omnivoice-studio

log_file = open("/tmp/omnivoice.log", "w")
subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log_file, stderr=log_file)

print("⏳ Waiting 15s for server startup...")
time.sleep(15)
!curl -sf http://localhost:3900/health || echo '❌ Server starting failed! Check /tmp/omnivoice.log'

In [ ]:
# 🎵 Step 4: Run Full Batch Generation (v1 to v5)
import os, time, json, urllib.request

BASE_OUTPUT_DIR = "/kaggle/working/outputs"
API_URL = "http://localhost:3900/v1/audio/speech"

def gen(text, voice="shimmer", model="tts-1", output_path="", speed=1.0, num_step=32, guidance_scale=2.0, instruct=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {"model": model, "voice": voice, "input": text, "response_format": "wav", "speed": speed, "num_step": num_step, "guidance_scale": guidance_scale}
    if instruct: payload["instruct"] = instruct
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data, headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            print(f"  ✅ {os.path.basename(output_path)} ({len(content)//1024} KB in {time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")

print("🚀 Starting Batch Generation (v1-v5)...")

# V1 Test
print("\n--- V1 Test ---")
gen("Hello! OmniVoice Studio on Kaggle GPU.", voice="alloy", output_path=f"{BASE_OUTPUT_DIR}/v1_test/v1_test_alloy.wav")
gen("Testing echo voice preset.", voice="echo", output_path=f"{BASE_OUTPUT_DIR}/v1_test/v1_test_echo.wav")

# V2 McCullen
print("\n--- V2 McCullen Nanomites ---")
mccullen = "Nanomites... programmed to devour metal, steel, flesh. But more importantly, they can be programmed to stop. The real-world applications are endless... So, you tell me... is it working?"
gen(mccullen, voice="alloy", output_path=f"{BASE_OUTPUT_DIR}/v2_mccullen/v2_mccullen_alloy.wav")
gen(mccullen, voice="onyx", output_path=f"{BASE_OUTPUT_DIR}/v2_mccullen/v2_mccullen_onyx.wav")
gen(mccullen, voice="shimmer", output_path=f"{BASE_OUTPUT_DIR}/v2_mccullen/v2_mccullen_shimmer.wav")

# V3 Movies
print("\n--- V3 Movie Dialogues ---")
gen("Kitne aadmi the? ... Do? ... Aur tum teen! ... Phir bhi wapas aa gaye... Khaali haath!", voice="alloy", output_path=f"{BASE_OUTPUT_DIR}/v3_movies/v3_sholay_gabbar_hindi.wav")
gen("আম্মাজান! আপনি শুধু একটা বার নির্দেশ দেন, আজ পুরো পৃথিবীকে আমি আপনার পায়ের নিচে এনে হাজির করব!", voice="onyx", output_path=f"{BASE_OUTPUT_DIR}/v3_movies/v3_ammajan_manna_bangla.wav")
gen("My name is Maximus Decimus Meridius, commander of the Armies of the North, General of the Felix Legions, and loyal servant to the true emperor, Marcus Aurelius. Father to a murdered son, husband to a murdered wife. And I will have my vengeance, in this life or the next.", voice="shimmer", output_path=f"{BASE_OUTPUT_DIR}/v3_movies/v3_gladiator_maximus_english.wav")

# V4 Deep Male
print("\n--- V4 Deep Male Voices ---")
deep = "The darkness does not scare me. I have walked through fire, through war, through loss. And still I stand. Because I am not built from hope alone. I am forged from pain, from rage, from the silence between heartbeats."
for v in ["shimmer", "onyx", "echo", "alloy", "demo0001"]:
    gen(deep, voice=v, model="tts-1-hd", num_step=32, speed=0.9, output_path=f"{BASE_OUTPUT_DIR}/v4_deep_male/v4_deep_male_hd_{v}.wav")

# V5 Ultra Human & Emotions
print("\n--- V5 Ultra Human & Emotions (Shimmer) ---")
gen("The darkness does not scare me anymore. I have walked through fire... through war... through loss. And still... I stand.", voice="shimmer", speed=0.85, num_step=32, guidance_scale=2.5, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_shimmer_slow_hq.wav")
gen("No! NO! I will NOT let this happen again! You hear me?! I have given EVERYTHING! Every last drop of blood, every ounce of strength!", voice="shimmer", speed=1.05, num_step=32, guidance_scale=3.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_rage.wav")
gen("She's gone... I... I can't believe she's actually gone. We had so many plans... so many dreams we never... We never got to live them out. And now... this silence... it's deafening.", voice="shimmer", speed=0.85, num_step=32, guidance_scale=2.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_sadness.wav")
gen("Ha ha ha! Oh man, you should have seen the look on his face! I swear, I haven't laughed this hard in years! Oh god, my stomach hurts from laughing!", voice="shimmer", speed=1.0, num_step=32, guidance_scale=2.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_laughter.wav")
gen("Listen carefully... I'm only going to say this once. They're watching us. Don't turn around. Just keep walking, and when I say run... you run.", voice="shimmer", speed=0.8, num_step=32, guidance_scale=1.5, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_whisper.wav")
gen("Ugh... I feel terrible. This cold is killing me. Can barely breathe through my nose... My throat is raw and scratchy. Every time I try to speak... it hurts.", voice="shimmer", speed=0.9, num_step=32, guidance_scale=2.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_sick.wav")

print("\n🎉 ALL SAMPLES GENERATED SUCCESSFULLY!")

In [ ]:
# 📦 Step 5: Zip all outputs for 1-click Download from Kaggle Output tab
!apt-get install -y zip
!zip -r /kaggle/working/omnivoice_outputs_v1_to_v5.zip /kaggle/working/outputs
print("\n✅ DOWNLOAD READY!")
print("📁 Kaggle Output Tab -> Download 'omnivoice_outputs_v1_to_v5.zip'")